# Week 4 示例：BiLSTM 与 CNN 特征图

- 作者：共享仓库示例
- 日期：2026-07-14
- 来源：`暑期居家集训学习计划.md` → Week 4 → RNN/Transformer/BERT 与 CNN/迁移学习
- 适用周次：Week 4
- 分类：PyTorch / NLP / Computer Vision
- 关键词：BiLSTM、注意力池化、ResNet、forward hook
- 运行环境：Python 3.10+、PyTorch、torchvision
- 适合读者：已经理解 Tensor、神经网络和基本训练流程的初学者

## 学习目标

1. 看懂 Embedding、双向 LSTM 和注意力池化的连接。
2. 用 forward hook 读取 CNN 中间层特征。

> 这里只演示模型定义、前向传播和 hook，不下载 BERT 权重，也不进行完整训练。

这份 Notebook 的重点是先看懂张量形状和模块连接，再把模型接到真实数据集。

In [ ]:
try:
    import torch
    import torch.nn as nn
except ImportError:
    torch = None
    nn = None
    print('PyTorch 未安装，跳过 PyTorch 单元。')
else:
    torch.manual_seed(42)
    print('PyTorch:', torch.__version__)

## 1. BiLSTM 文本分类器

下面保留《实验室新生暑期居家集训学习计划》中的 Embedding、双向 LSTM、注意力加权池化和分类层。随机输入只用于检查前向传播的张量形状。

In [ ]:
if torch is None:
    print('请先安装 PyTorch，再运行这一节。')
else:
    class BiLSTMClassifier(nn.Module):
        def __init__(self, vocab_size, embed_dim=32, hidden_dim=32, num_classes=2):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
            self.lstm = nn.LSTM(
                embed_dim, hidden_dim, batch_first=True, bidirectional=True
            )
            self.dropout = nn.Dropout(0.3)
            self.attention = nn.Linear(hidden_dim * 2, 1)
            self.classifier = nn.Linear(hidden_dim * 2, num_classes)

        def forward(self, input_ids):
            embedded = self.dropout(self.embedding(input_ids))
            output, _ = self.lstm(embedded)
            weights = torch.softmax(self.attention(output), dim=1)
            context = (weights * output).sum(dim=1)
            return self.classifier(self.dropout(context))

    model = BiLSTMClassifier(vocab_size=1000)
    input_ids = torch.randint(1, 1000, (4, 12))
    logits = model(input_ids)
    print('input shape:', tuple(input_ids.shape))
    print('logits shape:', tuple(logits.shape))

## 2. ResNet hook：读取中间层特征

《实验室新生暑期居家集训学习计划》使用预训练 ResNet50。为了避免下载权重，这里用 `weights=None` 的 ResNet18 做同样的 hook 演示；真实实验时再替换为预训练模型和真实图像。

In [ ]:
try:
    import torchvision.models as models
except Exception as exc:
    print('torchvision 当前不可用，跳过 CNN hook 示例：', exc)
else:
    cnn = models.resnet18(weights=None)
    cnn.eval()
    feature_maps = {}

    def hook_fn(module, inputs, output):
        feature_maps['layer1'] = output.detach()

    hook = cnn.layer1.register_forward_hook(hook_fn)
    input_tensor = torch.randn(1, 3, 224, 224)
    with torch.no_grad():
        output = cnn(input_tensor)
    print('model output:', tuple(output.shape))
    print('layer1 feature map:', tuple(feature_maps['layer1'].shape))
    hook.remove()

## 小结

完整 Week 4 还需要补充 tokenizer、BERT 微调、训练循环、迁移学习数据预处理和特征图可视化。